In [137]:
import pickle
import numpy as np
with open("signal.pkl", "rb") as f:
    signal = pickle.load(f)
with open("levels.pkl", "rb") as f:
    levels = pickle.load(f)
with open("seq_band.pkl", "rb") as f:
    seq_band = pickle.load(f)
with open("short_dwell_pen.pkl", "rb") as f:
    short_dwell_penalty = pickle.load(f)
with open("core_method.pkl", "rb") as f:
    core_method = pickle.load(f)

In [138]:
import plotly.graph_objects as go

fig = go.Figure()
fig['layout']['yaxis']['autorange'] = "reversed"

for x in range(len(levels[:1000])):
    y1, y2 = seq_band[:, x]
    fig.add_trace(
        go.Scatter(x=[x,x], y=[y1,y2], name=None, showlegend=False)
    )

fig.show()

In [139]:
np.cumsum(np.diff(seq_band, axis=0)[0])

array([    46,     95,    163, ..., 915206, 915277, 915339])

# seq_banded_dp

## 1. Prepare the score and traceback array
Rather than a 2D table, Remora uses a 1D array to store all scores. To access specific "columns", an offset array is calculated: 


```python
# Prepare for banded forwards pass followed by traceback
base_offsets_raw = np.cumsum(np.diff(seq_band, axis=0)[0])
band_len = base_offsets_raw[base_offsets_raw.shape[0] - 1]
if band_len > np.iinfo(np.uint32).max:
    raise RemoraError(
        "Dynamic programming search space too large. Read likely "
        "contains large deletions."
    )
base_offsets = np.empty(seq_band.shape[1] + 1, dtype=np.uint32)
base_offsets[0] = 0
base_offsets[1:] = base_offsets_raw

# all_scores array holds current score at every sequence by signal position
# within the band
all_scores = np.empty(band_len, dtype=np.float32)
# traceback contains one value for each position in the band and contains
# the number of stays from this position until the next move back to the
# previous base/level.
traceback = np.empty(band_len, dtype=np.int32)
```

## 2. Perform the forward pass to calculate the scores:

```python
# Execute forward pass filling all_scores and traceback
banded_forward_dp(
    all_scores,
    traceback,
    signal,
    levels,
    seq_band,
    base_offsets,
    short_dwell_penalty,
    core_method,
)
```

## 3. Perform the traceback:

```python
# path is primary return value as described in the return type
path = np.empty(levels.shape[0] + 1, dtype=np.int32)
# perform traceback and full path
banded_traceback(path, seq_band, base_offsets, traceback)
```

## Code Snippets:

In [140]:
base_offsets_raw = np.cumsum(np.diff(seq_band, axis=0)[0])
band_len = base_offsets_raw[base_offsets_raw.shape[0] - 1]
if band_len > np.iinfo(np.uint32).max:
    raise Exception(
        "Dynamic programming search space too large. Read likely "
        "contains large deletions."
    )
base_offsets = np.empty(seq_band.shape[1] + 1, dtype=np.uint32)
base_offsets[0] = 0
base_offsets[1:] = base_offsets_raw
print("The sum of all band lengths:", band_len)
print(base_offsets, base_offsets.shape)

The sum of all band lengths: 915339
[     0     46     95 ... 915206 915277 915339] (6903,)


In [141]:
# all_scores array holds current score at every sequence by signal position
# within the band
all_scores = np.empty(band_len, dtype=np.float32)
# traceback contains one value for each position in the band and contains
# the number of stays from this position until the next move back to the
# previous base/level.
traceback = np.empty(band_len, dtype=np.int32)

print("all_scores:", all_scores, all_scores.shape)
print("traceback:", traceback, traceback.shape)

all_scores: [9.7067239e-27 4.2134242e-41 9.7067239e-27 ... 2.9483189e-35 4.2134242e-41
 2.8929467e-35] (915339,)
traceback: [ 102188736      30068 1005633328 ...          0          0          0] (915339,)


# banded_forward_dp

## Initialize
Based on the given core_method string, choose the function that actually calculates the scores. Can be either *banded_forward_vit_step* or *banded_forward_dwell_penalty_step*:

```python
cdef core_func_ptr core_method_func
if core_method == REFINE_ALGO_VIT_NAME:
    core_method_func = banded_forward_vit_step
elif core_method == REFINE_ALGO_DWELL_PEN_NAME:
    core_method_func = banded_forward_dwell_penalty_step
else:
    raise RemoraError(
        f"Invalid core signal mapping refine method: {core_method}"
    )
```

## Calculate the scores for the first base:

```python
# compute first base forward scores
curr_bw = seq_band[1, 0]
# spoof previous scores to force stays through first base
prev_scores = np.full(curr_bw, HUGE_VALF, dtype=np.float32)
prev_scores[0] = 0
core_method_func(
    all_scores[:curr_bw],
    traceback[:curr_bw],
    prev_scores,
    levels[0],
    signal[:curr_bw],
    1,
    short_dwell_penalty,
)
prev_bw = curr_bw
prev_band_st = prev_offset = 0
```

## Compute the scores for all remaining sequence positions:

```python
# compute forward scores for all bases
for base_idx in range(1, levels.shape[0]):
    curr_band_st = seq_band[0, base_idx]
    curr_band_en = seq_band[1, base_idx]
    curr_bw = curr_band_en - curr_band_st
    curr_offset = base_offsets[base_idx]
    # compute all scores and traceback for this base
    core_method_func(
        all_scores[curr_offset:curr_offset + curr_bw],
        traceback[curr_offset:curr_offset + curr_bw],
        all_scores[prev_offset:prev_offset + prev_bw],
        levels[base_idx],
        signal[curr_band_st:curr_band_en],
        curr_band_st - prev_band_st,
        short_dwell_penalty,
    )
    prev_band_st = curr_band_st
    prev_bw = curr_bw
    prev_offset = curr_offset
```

# banded_forward_vit_step

Process one base using the [Viterbi](https://www.geeksforgeeks.org/viterbi-algorithm-for-hidden-markov-models-hmms/) path scoring algorithm  with squared error between signals and levels.

It takes the following arguments:
- the score array in the range of the current band: `all_scores[offset:offset+bandwidth]`
- the same subset from the traceback: `traceback[offset:offset+bandwidth]`
- the scores calculated in the previous iteration: `all_scores[prev.offset:prev.offset+prev.bandwidth]`
- the expected signal of the current base: `levels[currentindex]`
- the signal within the current band: `signal[band_start:band_end]`
- the difference between the current and previous band start: `band_start-prev.band_start`
- penalty scores for short dwells (UNUSED HERE; see in banded_forward_dwell_penalty_step)

## 1. calculate the score for the first band position

```python
# compute start position in band
if band_start_diff == 0:
    # if this is a "stay" band start, set invalid score and traceback
    curr_scores[0] = LARGE_SCORE + prev_scores[prev_scores.shape[0] - 1]
    curr_tb[0] = -1
else:
    # else compute move score for start of base band
    base_score = score(curr_level, curr_signal[0])
    curr_scores[0] = prev_scores[band_start_diff - 1] + base_score
    curr_tb[0] = 0
    # clip prev_scores to start at same position as curr_scores
    prev_scores = prev_scores[band_start_diff:]
```

Trim the last position from the previous if current and previous bands have the same shape:

```python
# if base bands are the same
if prev_scores.shape[0] == curr_scores.shape[0]:
    prev_scores = prev_scores[:prev_scores.shape[0] - 1]
```

## 2. Compute the scores and traceback values 

For each band position where current and previous bands overlap:

```python
# compute scores where curr and prev base overlap
for band_pos in range(1, prev_scores.shape[0] + 1):
    base_score = score(curr_level, curr_signal[band_pos])
    move_score = prev_scores[band_pos - 1] + base_score
    stay_score = curr_scores[band_pos - 1] + base_score
    if move_score < stay_score:
        curr_scores[band_pos] = move_score
        curr_tb[band_pos] = 0
    else:
        curr_scores[band_pos] = stay_score
        curr_tb[band_pos] = curr_tb[band_pos - 1] + 1
```

For each band position where the current bands hangs over the previous one:

```python
    # stay through rest of the band
    for band_pos in range(prev_scores.shape[0] + 1, curr_scores.shape[0]):
        base_score = score(curr_level, curr_signal[band_pos])
        stay_score = curr_scores[band_pos - 1] + base_score
        curr_scores[band_pos] = stay_score
        curr_tb[band_pos] = curr_tb[band_pos - 1] + 1
```

In [174]:
def score(s, l):
    """Find squared difference between sample and level.

    Args:
        s (float): sample
        l (float): level
    """
    tmp = s - l
    return tmp * tmp

def banded_forward_vit_step(
    curr_scores: np.ndarray, 
    curr_tb: np.ndarray, 
    prev_scores: np.ndarray,
    curr_level: float, 
    curr_signal: np.ndarray, 
    band_start_diff: int
):
    LARGE_SCORE = 100
    if band_start_diff == 0:
        # if this is a "stay" band start, set invalid score and traceback
        curr_scores[0] = LARGE_SCORE + prev_scores[prev_scores.shape[0] - 1]
        curr_tb[0] = -1
    else:
        # else compute move score for start of base band
        base_score = score(curr_level, curr_signal[0])
        curr_scores[0] = prev_scores[band_start_diff - 1] + base_score
        curr_tb[0] = 0
        # clip prev_scores to start at same position as curr_scores
        prev_scores = prev_scores[band_start_diff:]
    # if base bands are the same
    if prev_scores.shape[0] == curr_scores.shape[0]:
        prev_scores = prev_scores[:prev_scores.shape[0] - 1]

    # compute scores where curr and prev base overlap
    for band_pos in range(1, prev_scores.shape[0] + 1):
        base_score = score(curr_level, curr_signal[band_pos])
        move_score = prev_scores[band_pos - 1] + base_score
        stay_score = curr_scores[band_pos - 1] + base_score
        if move_score < stay_score:
            curr_scores[band_pos] = move_score
            curr_tb[band_pos] = 0
        else:
            curr_scores[band_pos] = stay_score
            curr_tb[band_pos] = curr_tb[band_pos - 1] + 1

    # stay through rest of the band
    for band_pos in range(prev_scores.shape[0] + 1, curr_scores.shape[0]):
        base_score = score(curr_level, curr_signal[band_pos])
        stay_score = curr_scores[band_pos - 1] + base_score
        curr_scores[band_pos] = stay_score
        curr_tb[band_pos] = curr_tb[band_pos - 1] + 1

    return curr_scores, curr_tb

## Running the function for the first base

In [175]:
curr_bw = seq_band[1,0]

first_scores = all_scores[:curr_bw],
first_tb = traceback[:curr_bw],

prev_scores = np.full(curr_bw, 10e20)
prev_scores[0] = 0

first_level = levels[0]
first_signal = signal[:curr_bw]

first_band_start_diff = 1

print(curr_bw)
print(first_scores[0], first_scores[0].shape)
print(first_tb[0], first_tb[0].shape)
print(prev_scores, prev_scores.shape)
print(first_level)
print(first_signal, first_signal.shape)
print(first_band_start_diff)

46
[ 0.28620595  0.5889039   1.0199802   1.3171295   1.672075    1.9747729
  2.260979    2.5054646   2.7971165   3.0219738   3.4333246   3.4914825
  3.654108    3.8334875   4.3775687   4.612138    4.8516393   5.1599374
  5.4911485   5.8108006   6.0453696   6.489853    6.803802    7.100951
  7.3210297   7.694315    8.125391    8.3903265   8.751334    9.088402
  9.431377    9.628332    9.839007   10.136157   10.433307   10.730456
 11.097577   11.332146   11.737023   12.074091   12.202023   12.456631
 12.691199   12.751826   12.802885   12.809784  ] (46,)
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45] (46,)
[0.e+00 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21
 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21
 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21
 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21 1.e+21
 1.e+21 1.

In [176]:
first_scores_out, first_tb_out = banded_forward_vit_step(
    curr_scores = first_scores[0],
    curr_tb = first_tb[0],
    prev_scores = prev_scores,
    curr_level = first_level,
    curr_signal = first_signal,
    band_start_diff = first_band_start_diff
)

In [177]:
first_scores_out, first_tb_out

(array([ 0.28620595,  0.5889039 ,  1.0199802 ,  1.3171295 ,  1.672075  ,
         1.9747729 ,  2.260979  ,  2.5054646 ,  2.7971165 ,  3.0219738 ,
         3.4333246 ,  3.4914825 ,  3.654108  ,  3.8334875 ,  4.3775687 ,
         4.612138  ,  4.8516393 ,  5.1599374 ,  5.4911485 ,  5.8108006 ,
         6.0453696 ,  6.489853  ,  6.803802  ,  7.100951  ,  7.3210297 ,
         7.694315  ,  8.125391  ,  8.3903265 ,  8.751334  ,  9.088402  ,
         9.431377  ,  9.628332  ,  9.839007  , 10.136157  , 10.433307  ,
        10.730456  , 11.097577  , 11.332146  , 11.737023  , 12.074091  ,
        12.202023  , 12.456631  , 12.691199  , 12.751826  , 12.802885  ,
        12.809784  ], dtype=float32),
 array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
        34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45], dtype=int32))

## Running the function for the first base

In [178]:
base_idx = 1

curr_band_st = seq_band[0, base_idx]
curr_band_en = seq_band[1, base_idx]
curr_bw = curr_band_en - curr_band_st
curr_offset = base_offsets[base_idx]

prev_band_st = seq_band[0, base_idx-1]
prev_band_en = seq_band[1, base_idx-1]
prev_bw = prev_band_en - prev_band_st
prev_offset = base_offsets[base_idx-1]

curr_scores = all_scores[curr_offset:curr_offset + curr_bw]
curr_tb = traceback[curr_offset:curr_offset + curr_bw]

prev_scores = all_scores[prev_offset:prev_offset + prev_bw]

curr_level = levels[base_idx]
curr_signal = signal[curr_band_st:curr_band_en]

curr_band_start_diff = curr_band_st - prev_band_st

print(curr_scores, curr_scores.shape)
print(curr_tb, curr_tb.shape)
print(prev_scores, prev_scores.shape)
print(curr_level)
print(curr_signal, curr_signal.shape)
print(curr_band_start_diff)

[ 0.5889039  1.0199802  1.3171295  1.672075   1.9747729  2.260979
  2.5054646  2.7971165  3.0219738  3.4333246  3.4914825  3.654108
  3.8334875  4.3775687  4.612138   4.8516393  5.1599374  5.4911485
  5.8108006  6.0453696  6.489853   6.803802   7.100951   7.3210297
  7.694315   8.125391   8.3903265  8.751334   9.088402   9.431377
  9.628332   9.839007  10.136157  10.433307  10.730456  11.097577
 11.332146  11.737023  12.074091  12.202023  12.456631  12.691199
 12.751826  12.802885  12.809784  13.5832405 14.545174  16.452303
 19.464619 ] (49,)
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48] (49,)
[ 0.28620595  0.5889039   1.0199802   1.3171295   1.672075    1.9747729
  2.260979    2.5054646   2.7971165   3.0219738   3.4333246   3.4914825
  3.654108    3.8334875   4.3775687   4.612138    4.8516393   5.1599374
  5.4911485   5.8108006   6.0453696   6.489853    6.803802    7.100951
  7.321

In [179]:
curr_scores_out, curr_tb_out = banded_forward_vit_step(
    curr_scores = curr_scores,
    curr_tb = curr_tb,
    prev_scores = prev_scores,
    curr_level = curr_level,
    curr_signal = curr_signal,
    band_start_diff = curr_band_start_diff
)

In [180]:
print(curr_scores_out, curr_tb_out)

[ 0.5889039  1.0199802  1.3171295  1.672075   1.9747729  2.260979
  2.5054646  2.7971165  3.0219738  3.4333246  3.4914825  3.654108
  3.8334875  4.3775687  4.612138   4.8516393  5.1599374  5.4911485
  5.8108006  6.0453696  6.489853   6.803802   7.100951   7.3210297
  7.694315   8.125391   8.3903265  8.751334   9.088402   9.431377
  9.628332   9.839007  10.136157  10.433307  10.730456  11.097577
 11.332146  11.737023  12.074091  12.202023  12.456631  12.691199
 12.751826  12.802885  12.809784  13.5832405 14.545174  16.452303
 19.464619 ] [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48]


# banded_forward_dwell_penalty_step

This calculates the scores for one base using with an introduced penalty for short dwell times ('*It's better if each base is represented by at least N signal points.*'). For this it uses the **short_dwell_penalty** array:
- penalty scores for short dwells

All other arguments are the same as before:

- the score array in the range of the current band: `all_scores[offset:offset+bandwidth]`
- the same subset from the traceback: `traceback[offset:offset+bandwidth]`
- the scores calculated in the previous iteration: `all_scores[prev.offset:prev.offset+prev.bandwidth]`
- the expected signal of the current base: `levels[currentindex]`
- the signal within the current band: `signal[band_start:band_end]`
- the difference between the current and previous band start: `band_start-prev.band_start`

## Loop over each band position

```python

```